Describing Welfare
==================

**Author:** Ethan Ligon



Constructing a consumption aggregate, and then describing the distribution
of it, before modelling anything.



## Reading



Deaton is in your `reading/` folder on the hub, or [PDF](https://documents.worldbank.org/curated/en/203811547671768139/pdf/133790-PUB.pdf).

-   Deaton, chs. 3 and 4
-   Deaton & Zaidi, *Guidelines for Constructing Consumption Aggregates* —
    the operational reference, and the one you will actually reach for
-   Deaton & Zaidi §§2–4 if you read nothing else



## Why consumption, not income



## The four components



## Own production and the valuation problem



## Durables and housing



## Deflation



## Adult equivalence



## Poverty measures



## Inequality



## Building the aggregate



`LSMS_Library` does the food side for you.  Start from what was acquired:



In [1]:
# Show tracebacks without the library's internal frames: the line that
# failed, and why.  Change Plain to Verbose if you ever want the rest.
%xmode Plain

import lsms_library as ll
import numpy as np, pandas as pd

ghana = ll.Country('GhanaLSS')
acq = ghana.food_acquired()
acq.head()

The index is $(t, i, j, u, s, \mathit{visit})$ — wave, household, item,
unit, source, visit — and the columns are `Expenditure`, `Quantity`,
`Price`.  The derived table collapses this to expenditure per household-item:



In [1]:
food = ghana.food_expenditures()
food.head()

Total food consumption per household is then a sum over items:



In [1]:
food_total = food.groupby(['t', 'i']).sum().squeeze()
food_total.groupby('t').describe().round(0)

## Unit values, and what they hide



Look at the unit values directly, for one item, in one round.



In [1]:
uv = (acq.Expenditure / acq.Quantity).rename('unit_value')
uv = uv.replace([np.inf, -np.inf], np.nan).dropna()

item = uv.xs('2016-17', level='t').groupby('j').size().idxmax()  # commonest item
x = uv.xs('2016-17', level='t').xs(item, level='j')
print(item, ':', len(x), 'observations')
x.groupby('u').describe().round(2)          # by unit of measure

Two lessons, both visible in that table.  Unit values vary enormously within
an item — ratios of 100 to 1 are common, and are misreported quantities,
not real price dispersion.  And they are only comparable *within* a unit of
measure, which is why `u` is in the index.



In [1]:
# The median is the robust summary; the mean is not.
x.groupby('u').agg(['median', 'mean', 'count']).round(2).head()

## Poverty, correctly weighted



In [1]:
sample = ghana.sample()

def fgt(c, w, z, alpha=0):
    """Weighted FGT_alpha.  c: consumption per adult equivalent; w: weights."""
    c, w = np.asarray(c, float), np.asarray(w, float)
    ok = np.isfinite(c) & np.isfinite(w)
    c, w = c[ok], w[ok]
    # Sum over the poor only.  Writing this as np.sum(w * gap**alpha) over
    # everybody looks equivalent, and is -- except at alpha=0, where
    # 0**0 == 1 and every household in the country counts as poor.
    poor = c < z
    return np.sum(w[poor] * ((z - c[poor]) / z) ** alpha) / np.sum(w)

wave = '2016-17'
c = food_total.xs(wave, level='t')
w = sample.xs(wave, level='t').weight.reindex(c.index)

z = c.quantile(0.25)          # a placeholder line; see the exercise
for a in (0, 1, 2):
    print(f"P_{a} = {fgt(c, w, z, a):.4f}")

# The line is an unweighted quartile, so the UNWEIGHTED headcount must come
# back as 0.25 by construction.  It is the one number here we know in
# advance, which makes it the right thing to check the code against.
print(f"unweighted P_0 = {fgt(c, np.ones(len(c)), z, 0):.4f}")
print(f"mean weight, poor = {w[c < z].mean():.3f}"
      f"   non-poor = {w[c >= z].mean():.3f}")

The unweighted headcount returns 0.2498, which is 0.25 up to the
discreteness of the sample — as it must be, since the line *is* the
unweighted lower quartile.  That is the check: it is the one quantity here
whose value you know before running the code, so it is the one that tells
you whether the code is right.  If it comes back as 1.0000, you have found
the $0^0$ trap.

The weighted headcount is 0.1537 — nine percentage points lower.  The
reason is in the line below it: poor households carry a mean weight of 0.62
against 1.13 for everyone else, because the design over-sampled them.
Session 1's lesson, arriving as a number you might otherwise have published.

None of which rescues the line itself.  A quartile of the distribution is
not a poverty line: it fixes the unweighted headcount at 0.25 by
construction, in every country and every year, which is a fact about
arithmetic rather than about Ghana.  A real line is
absolute — the cost of a fixed bundle — and does not move with the
distribution, and does not move when everyone gets poorer.  Fixing that is
the first exercise.



## Lorenz and Gini



In [1]:
def lorenz(c, w):
    c, w = np.asarray(c, float), np.asarray(w, float)
    ok = np.isfinite(c) & np.isfinite(w) & (c >= 0)
    c, w = c[ok], w[ok]
    order = np.argsort(c)
    c, w = c[order], w[order]
    p = np.cumsum(w) / np.sum(w)
    L = np.cumsum(w * c) / np.sum(w * c)
    return np.concatenate([[0], p]), np.concatenate([[0], L])

def gini(c, w):
    p, L = lorenz(c, w)
    return 1 - 2 * np.trapezoid(L, p)

print(f"Gini (food consumption, {wave}) = {gini(c, w):.3f}")

In [1]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 5))
for wv in ['1998-99', '2005-06', '2012-13', '2016-17']:
    cc = food_total.xs(wv, level='t')
    ww = sample.xs(wv, level='t').weight.reindex(cc.index)
    p, L = lorenz(cc, ww)
    ax.plot(p, L, label=f"{wv}  (G={gini(cc, ww):.2f})")
ax.plot([0, 1], [0, 1], 'k--', lw=0.8)
ax.set_xlabel('cumulative share of households')
ax.set_ylabel('cumulative share of food consumption')
ax.legend(frameon=False)
plt.show()

If two Lorenz curves cross, the ranking of the two distributions depends on
which inequality measure you chose, and no scalar will rescue you.  Look
before you summarize.



## One index, or several?



## Exercises



1.  The poverty line used above is fake.  Build a real one: take the food
    bundle consumed by households in the second and third deciles, price it at
    national median unit values, and scale it to 2,900 kcal per adult
    equivalent.  Recompute $P_0, P_1, P_2$.
2.  Deflate spatially.  Construct a regional Paasche index from the survey's
    own unit values, apply it, and report how much of the north–south poverty
    gradient survives.
3.  Vary $\theta$ in $c_i = C_i / A_i^\theta$ over $[0.5, 1.0]$ and plot
    the headcount ratio against it.  Over what range does the *ranking* of
    regions change?

